## Imports

In [21]:
import pandas as pd
import random
import os
import time
import openai
import anthropic
import google.generativeai as genai


ModuleNotFoundError: No module named 'openai'

## Valdation

### MCP 30 Samples

In [22]:
# Load your full MCP dataset
mcp_df = pd.read_csv("data/mcp_desc_all_nov_7_cleaned.csv")

# --- Define keywords (expanded sets) ---
bucket_1_keywords = [
    "search", "query", "retrieve", "scrape", "crawl", "research",
    "knowledge", "api", "data"
]
bucket_2_keywords = [
    "generate", "image", "video", "audio", "tts", "voice", "picture",
    "text", "story", "art", "music"
]
bucket_3_keywords = [
    "automation", "code", "execute", "browser", "workflow", "tool",
    "script", "process", "run", "system"
]

# Lowercase text for matching
mcp_df["text_lower"] = mcp_df["text_for_llm"].str.lower()

# Function to count keyword hits
def count_hits(text, keywords):
    return sum(1 for kw in keywords if kw in text)

# Assign each MCP to a bucket
buckets = []
for text in mcp_df["text_lower"]:
    counts = [
        count_hits(text, bucket_1_keywords),
        count_hits(text, bucket_2_keywords),
        count_hits(text, bucket_3_keywords)
    ]
    # Pick the bucket with the most hits
    max_hits = max(counts)
    if max_hits == 0:
        buckets.append("unclassified")
    else:
        # tie-break: default to automation bucket (index 2)
        bucket_idx = counts.index(max_hits) if counts.count(max_hits) == 1 else 2
        buckets.append(["retrieval", "generative", "automation"][bucket_idx])

mcp_df["bucket"] = buckets

# --- Sample 10 from each of the three buckets ---
samples = []
for label in ["retrieval", "generative", "automation"]:
    subset = mcp_df[mcp_df["bucket"] == label]
    if len(subset) >= 10:
        samples.append(subset.sample(n=10, random_state=41))
    else:
        samples.append(subset)  # if fewer than 10 exist

sampled_df = pd.concat(samples)

# --- Save final 30-row dataset ---
sampled_df = sampled_df[
    ["title", "url", "uploaded_clean", "text_for_llm", "len_text", "bucket"]
]
sampled_df = sampled_df.drop(columns=["uploaded_clean", "len_text"])
sampled_df.to_csv("data/mcp_30_sample.csv", index=False)


### O*NET Data

In [20]:
# Load the O*NET CSV
onet_df = pd.read_csv("../onet_hierarchy/onet_tasks_dwa_iwa_gwa.csv")

# Keep only the columns relevant for the cascading sheet
onet_hierarchy = onet_df[[
    "gwa_title", "iwa_title", "dwa_title", "task"
]].drop_duplicates().reset_index(drop=True)

# Save for Google Sheets use
onet_hierarchy.to_csv("data/onet_options_sheet.csv", index=False)


## Classifying Scripts

### Prompts

In [ ]:
OCC_RELEVANCE_PROMPT = """
The following is a description and list of use cases of an AI Model Context Protocol (MCP) server — a plugin-like system 
that lets AI assistants access external tools, APIs, or data sources to perform real-world tasks.

<mcp_description>
{desc}
</mcp_description>

Your job is to answer this question about the above MCP server:
<question>
Does this MCP server perform or enable an occupationally relevant activity — that is, something a human could reasonably be paid to do within the economy?
</question>

You MUST answer either “Yes” or “No.” Provide your reasoning and answer to the above question in the following format and nothing else:

<thinking>
1–2 sentences explaining your reasoning.
</thinking>
<answer>
Yes or No
</answer>
"""


HIERARCHY_PROMPT_TEMPLATE_GWAS = """
The following is a description and list of use cases of an AI Model Context Protocol (MCP) server — a plugin-like system 
that lets AI assistants access external tools, APIs, or data sources to perform real-world tasks.

<mcp_description>
{desc}
</mcp_description>

Below is a list of General Work Activities (GWAs) from the O*NET occupational task and work activities hierarchy:
{options_list}

Your job is to answer this question about the above MCP server:
<question>
Which GWA from the above list best describes the primary economic activity performed or enabled by this MCP server?
</question>

If none clearly apply, respond with “None” — do not hesitate to respond with “None” when uncertain.

You MUST provide your reasoning and answer to the above question in the exact format below — nothing else:

<thinking>
1–2 sentences explaining your reasoning.
</thinking>
<answer>
Exact GWA from the list above, or None.
</answer>
"""


HIERARCHY_PROMPT_TEMPLATE_IWAS = """
The following is a description and list of use cases of an AI Model Context Protocol (MCP) server — a plugin-like system 
that lets AI assistants access external tools, APIs, or data sources to perform real-world tasks.

<mcp_description>
{desc}
</mcp_description>

Below is a list of Intermediate Work Activities (IWAs) from the O*NET occupational tasks and work activities hierarchy,
corresponding to the previously selected General Work Activity: "{prev_level_choice}".
{options_list}

Your job is to answer this question about the above MCP server:
<question>
Which IWA from the above list best describes the primary economic activity performed or enabled by this MCP server?
</question>

You MUST provide your reasoning and answer to the above question in the exact format below — nothing else:

<thinking>
1–2 sentences explaining your reasoning.
</thinking>
<answer>
Exact IWA from the list above.
</answer>
"""


HIERARCHY_PROMPT_TEMPLATE_DWAS = """
The following is a description and list of use cases of an AI Model Context Protocol (MCP) server — a plugin-like system 
that lets AI assistants access external tools, APIs, or data sources to perform real-world tasks.

<mcp_description>
{desc}
</mcp_description>

Below is a list of Detailed Work Activities (DWAs) from the O*NET occupational tasks and work activities hierarchy,
corresponding to the previously selected Intermediate Work Activity: "{prev_level_choice}".
{options_list}

Your job is to answer this question about the above MCP server:
<question>
Which DWA from the above list best describes the primary economic activity performed or enabled by this MCP server?
</question>

You MUST provide your reasoning and answer to the above question in the exact format below — nothing else:

<thinking>
1–2 sentences explaining your reasoning.
</thinking>
<answer>
Exact DWA from the list above.
</answer>
"""


HIERARCHY_PROMPT_TEMPLATE_TASKS = """
The following is a description and list of use cases of an AI Model Context Protocol (MCP) server — a plugin-like system 
that lets AI assistants access external tools, APIs, or data sources to perform real-world tasks.

<mcp_description>
{desc}
</mcp_description>

Below is a list of Tasks from the O*NET occupational tasks and work activities hierarchy,
corresponding to the previously selected Detailed Work Activity: "{prev_level_choice}".
{options_list}

Your job is to answer this question about the above MCP server:
<question>
Which Task or Tasks (choose up to three) from the above list best describes the primary economic activities performed or enabled by this MCP server?
</question>

You MUST provide your reasoning and answer to the above questionin the exact format below — nothing else:

<thinking>
1–2 sentences explaining your reasoning.
</thinking>
<answer>
List up to three exact Tasks from the list above, separated by semicolons.
</answer>
"""


### 30 MCP Test

In [ ]:
# === API KEYS ===
openai.api_key = os.getenv("OPENAI_API_KEY")
anth_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# === CONFIGURATION ===
INPUT_CSV = "data/mcp_sample_30.csv"  # update to your actual CSV
OUTPUT_DIR = "data/results/llm_outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODELS = {
    "claude-sonnet-4.5": {"provider": "anthropic", "model": "claude-3-5-sonnet-20241022"},
    "gpt-4.1": {"provider": "openai", "model": "gpt-4.1"},
    "gemini-2.5-pro": {"provider": "google", "model": "gemini-2.5-pro"},
}

# === PROMPT TEMPLATES ===
# (paste your finalized ones here exactly as defined)

# === HELPER FUNCTION ===
def call_model(provider, model_name, prompt, max_tokens=600):
    """Unified interface for OpenAI, Anthropic, and Google Gemini models."""
    if provider == "openai":
        response = openai.ChatCompletion.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
        )
        return response.choices[0].message.content

    elif provider == "anthropic":
        response = anth_client.messages.create(
            model=model_name,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.content[0].text

    elif provider == "google":
        model = genai.GenerativeModel(model_name)
        response = model.generate_content(prompt)
        return response.text

    else:
        raise ValueError(f"Unsupported provider: {provider}")

# === LOAD INPUT ===
df = pd.read_csv(INPUT_CSV)

# === PIPELINE ===
for model_key, cfg in MODELS.items():
    print(f"\n=== Running {model_key} ===")
    results = []

    for idx, row in df.iterrows():
        desc = row["text_for_llm"]

        # --- Step 0: Occupational Relevance ---
        prompt_occ = OCC_RELEVANCE_PROMPT.format(desc=desc)
        out_occ = call_model(cfg["provider"], cfg["model"], prompt_occ)
        is_relevant = "Yes" in out_occ.split("<answer>")[-1]

        rec = {
            "model": model_key,
            "index": idx,
            "title": row.get("title", ""),
            "occ_relevance_output": out_occ.strip(),
        }

        if not is_relevant:
            results.append(rec)
            continue

        # --- Step 1: GWA ---
        options_gwa = "\n".join(gwa_list)  # you'll insert your actual list
        prompt_gwa = HIERARCHY_PROMPT_TEMPLATE_GWAS.format(desc=desc, options_list=options_gwa)
        out_gwa = call_model(cfg["provider"], cfg["model"], prompt_gwa)
        gwa_choice = out_gwa.split("<answer>")[-1].strip()
        rec["gwa_output"] = out_gwa.strip()

        # --- Step 2: IWA ---
        options_iwa = "\n".join(iwa_map[gwa_choice])  # assuming you have dicts keyed by GWA
        prompt_iwa = HIERARCHY_PROMPT_TEMPLATE_IWAS.format(
            desc=desc,
            options_list=options_iwa,
            prev_level_choice=gwa_choice,
        )
        out_iwa = call_model(cfg["provider"], cfg["model"], prompt_iwa)
        iwa_choice = out_iwa.split("<answer>")[-1].strip()
        rec["iwa_output"] = out_iwa.strip()

        # --- Step 3: DWA ---
        options_dwa = "\n".join(dwa_map[iwa_choice])
        prompt_dwa = HIERARCHY_PROMPT_TEMPLATE_DWAS.format(
            desc=desc,
            options_list=options_dwa,
            prev_level_choice=iwa_choice,
        )
        out_dwa = call_model(cfg["provider"], cfg["model"], prompt_dwa)
        dwa_choice = out_dwa.split("<answer>")[-1].strip()
        rec["dwa_output"] = out_dwa.strip()

        # --- Step 4: Tasks ---
        options_task = "\n".join(task_map[dwa_choice])
        prompt_task = HIERARCHY_PROMPT_TEMPLATE_TASKS.format(
            desc=desc,
            options_list=options_task,
            prev_level_choice=dwa_choice,
        )
        out_task = call_model(cfg["provider"], cfg["model"], prompt_task)
        rec["task_output"] = out_task.strip()

        results.append(rec)
        time.sleep(1)

    pd.DataFrame(results).to_csv(f"{OUTPUT_DIR}/{model_key}_outputs.csv", index=False)
    print(f"✅ Saved {model_key}_outputs.csv")

print("\n=== All models completed successfully ===")
